# Baixar vídeos do S3

Autor:  Viviane da Rosa Sommer

Atualizado: 24/07/2025

Objetivo: O objetivo desse código é automatizar o download de todos os arquivos de uma pasta específica em um bucket S3 da AWS para um diretório local, mantendo a estrutura de diretórios original. Ele utiliza credenciais armazenadas em um arquivo de configuração para autenticar no S3 e faz o download recursivo dos arquivos de uma pasta definida, facilitando a sincronização de dados entre o S3 e o ambiente local.

## Importações necessárias

In [ ]:
import boto3
import os
import json
from typing import NoReturn

## Declaração de Constantes e Modelos

In [ ]:
with open("config.json", 'r') as f:
    config = json.load(f)
data_config = config[0]

s3 = boto3.client(
    's3',
    aws_access_key_id=data_config["aws_access_id"],
    aws_secret_access_key=data_config["secret_key"]
)

bucket_name = 'analise-dados'
s3_folder = "projeto-ia-submarina/ia-descomissionamento/passagem-conhecimento/"
local_dir = "../Passagem-Conhecimento-Descom/Teste/videos"
os.makedirs(local_dir, exist_ok=True)

## Funções necessárias

In [ ]:
def download_s3_folder(bucket_name: str, s3_folder: str, local_dir: str) -> NoReturn:
    """
    Download all files from a folder in S3 to a local directory.

    Args:
        bucket_name (str): S3 bucket name.
        s3_folder (str): Folder path in the S3 bucket.
        local_dir (str): Local directory to save the files.
    """
    paginator = s3.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=s3_folder)

    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                print(obj)
                s3_file_path = obj['Key']
                relative_path = os.path.relpath(s3_file_path, s3_folder)
                if relative_path in ('.', '..'):
                    continue
                local_file_path = os.path.join(local_dir, relative_path)
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
                s3.download_file(bucket_name, s3_file_path, local_file_path)

## Baixar vídeos

In [ ]:
download_s3_folder(bucket_name, s3_folder, local_dir)